# Bevölkerungsdaten 2011-2026

Dieses Notebook lädt die BFS-Bevölkerungsdaten über die PXWeb-API.
Der Zeitraum wurde auf 2011 bis 2026 eingeschränkt.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

In [ ]:
DATA_DIR = Path("../../data")
RAW_DIR = Path(DATA_DIR / "raw")
INTERIM_DIR = Path(DATA_DIR / "interim")

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)
INTERIM_DIR.mkdir(exist_ok=True)

START_YEAR = 2011
END_YEAR = 2026

PXWEB_API_URL = (
    "https://www.pxweb.bfs.admin.ch/api/v1/de/"
    "px-x-0102020000_104/px-x-0102020000_104.px"
)

RAW_FILE = RAW_DIR / f"bevoelkerung_raw_{START_YEAR}_{END_YEAR}.csv"
OUTPUT_FILE = INTERIM_DIR / f"bevoelkerung_{START_YEAR}_{END_YEAR}.csv"

## Metadaten laden

Die Metadaten enthalten die verfügbaren Jahre und die Codes für Kantone, Alter und demografische Komponenten.

In [ ]:
metadata_response = requests.get(PXWEB_API_URL, timeout=30)
metadata_response.raise_for_status()
metadata = metadata_response.json()

variables = {variable["code"]: variable for variable in metadata["variables"]}
print(f'keys: {variables.keys()}')
print(f'values: {variables.values()}')

In [ ]:
available_years = variables["Jahr"]["values"]
selected_years = [
    year for year in available_years
    if START_YEAR <= int(year) <= END_YEAR
]

print(f"Verfügbare Jahre im Zeitraum: {selected_years[0]}-{selected_years[-1]}")
print(f"Anzahl Jahre: {len(selected_years)}")

## Query definieren

Für die Analyse brauchen wir primär den Bevölkerungsbestand nach Jahr, Kanton und Alter. Deshalb werden Staatsangehörigkeit und Geschlecht auf Total gesetzt und der Bestand am 31. Dezember gewählt.

In [ ]:
def values_by_text(variable_code: str, wanted_texts: set[str]) -> list[str]:
    variable = variables[variable_code]
    return [
        value
        for value, text in zip(variable["values"], variable["valueTexts"])
        if text in wanted_texts
    ]


nationality_total = values_by_text(
    "Staatsangehörigkeit (Kategorie)",
    {"Staatsangehörigkeit (Kategorie) - Total"},
)
gender_total = values_by_text(
    "Geschlecht",
    {"Geschlecht - Total"},
)
population_end_of_year = values_by_text(
    "Demografische Komponente",
    {"Bestand am 31. Dezember"},
)

print(nationality_total, gender_total, population_end_of_year)

In [ ]:
query = {
    "query": [
        {
            "code": "Jahr",
            "selection": {"filter": "item", "values": selected_years},
        },
        {
            "code": "Kanton",
            "selection": {"filter": "all", "values": ["*"]},
        },
        {
            "code": "Staatsangehörigkeit (Kategorie)",
            "selection": {"filter": "item", "values": nationality_total},
        },
        {
            "code": "Geschlecht",
            "selection": {"filter": "item", "values": gender_total},
        },
        {
            "code": "Alter",
            "selection": {"filter": "all", "values": ["*"]},
        },
        {
            "code": "Demografische Komponente",
            "selection": {"filter": "item", "values": population_end_of_year},
        },
    ],
    "response": {"format": "CSV"},
}

query

## Download

In [ ]:
def download_pxweb_csv(url: str, query: dict, target: Path) -> None:
    if target.exists():
        return

    response = requests.post(url, json=query, timeout=120)
    response.raise_for_status()
    target.write_bytes(response.content)

download_pxweb_csv(PXWEB_API_URL, query, RAW_FILE)

In [ ]:
df_population = pd.read_csv(RAW_FILE, encoding="cp1252")
df_population.head()

In [ ]:
df_population.to_csv(OUTPUT_FILE, index=False)

## Validierung

In [ ]:
print(df_population.shape)
print(df_population["Kanton"].nunique(), "Kantone inkl. Schweiz und ohne Angabe")
df_population.head()